# MERMAID annotations snapshot

Static exploration of `mermaid_confirmed_annotations.parquet`: schema, grouped corpus stats, label × growth-form overlap, and region-blocked stratified train/val holdout.

Default path matches `MermaidDataset`. Requires AWS credentials and `uv sync --extra notebooks`.


In [1]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

import boto3
import hvplot.pandas  # noqa: F401
import ibis
import pandas as pd
from great_tables import GT
from IPython.display import display

_root = Path.cwd().resolve()
for _candidate in (_root, *_root.parents):
    if (_candidate / "mermaidseg").is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

_main = (
    Path(
        subprocess.check_output(
            ["git", "rev-parse", "--git-common-dir"], cwd=_candidate, text=True
        ).strip()
    )
    .resolve()
    .parent
)
for _line in (_main / ".env").read_text().splitlines():
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        os.environ.setdefault(_k.strip(), _v.strip().strip("\"'"))

print(f"AWS_PROFILE: {os.getenv('AWS_PROFILE', '(default)')}")
print(boto3.client("sts").get_caller_identity()["Arn"])

AWS_PROFILE: mermaid-core
arn:aws:sts::554812291621:assumed-role/AWSReservedSSO_SageMaker_8be7db0b0a7583db/lauren@datamermaid.org


In [2]:
ibis.options.interactive = True

con = ibis.duckdb.connect()
con.raw_sql("""
  INSTALL httpfs;
  LOAD httpfs;
""")
con.raw_sql("CREATE OR REPLACE SECRET s3 (TYPE S3, PROVIDER CREDENTIAL_CHAIN)")

In [3]:
ANNOTATIONS_PATH = os.getenv(
    "MERMAID_ANNOTATIONS_PATH",
    "s3://coral-reef-training/mermaid/mermaid_confirmed_annotations.parquet",
)
HOLDOUT_FRACTION = 0.1
HOLDOUT_SEED = 42
GROUP_COLS = (
    "region_name",
    "benthic_attribute_name",
    "benthic_attribute_id",
    "growth_form_name",
)

ANNOTATIONS_PATH

's3://coral-reef-training/mermaid/mermaid_confirmed_annotations.parquet'

## Load annotations

In [4]:
ann = con.read_parquet(ANNOTATIONS_PATH)

required = {
    "image_id",
    "benthic_attribute_name",
    "benthic_attribute_id",
    "growth_form_name",
    "region_name",
}
missing = required - set(ann.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

ann = ann.mutate(
    image_id=ann.image_id.cast("string"),
    benthic_attribute_name=ann.benthic_attribute_name.cast("string"),
    benthic_attribute_id=ann.benthic_attribute_id.cast("string"),
    growth_form_name=ann.growth_form_name.cast("string"),
    region_name=ann.region_name.cast("string"),
)

total_annotations = ann.count().execute()
total_images = ann.image_id.nunique().execute()

In [5]:
schema_df = pd.DataFrame(
    [
        {
            "column": col,
            "dtype": str(ann[col].type()),
            "non_null": int(ann[col].count().execute()),
            "nunique": int(ann[col].nunique().execute()),
        }
        for col in ann.columns
    ]
)

summary_df = pd.DataFrame(
    [
        {
            "annotations": total_annotations,
            "images": total_images,
            "benthic_attribute_names": int(ann.benthic_attribute_name.nunique().execute()),
            "benthic_attribute_ids": int(ann.benthic_attribute_id.nunique().execute()),
            "growth_forms": int(ann.growth_form_name.nunique().execute()),
            "regions": int(ann.region_name.nunique().execute()),
        }
    ]
)

In [6]:
display(GT(summary_df).tab_header(title="MERMAID corpus snapshot", subtitle=ANNOTATIONS_PATH))

GT(_tbl_data=   annotations  images  benthic_attribute_names  benthic_attribute_ids  \
0       415825   16633                      251                    251   

   growth_forms  regions  
0            11        3  , _body=<great_tables._gt_data.Body object at 0x143ef5f40>, _boxhead=Boxhead([ColInfo(var='annotations', type=<ColInfoTypeEnum.default: 1>, column_label='annotations', column_align='right', column_width=None), ColInfo(var='images', type=<ColInfoTypeEnum.default: 1>, column_label='images', column_align='right', column_width=None), ColInfo(var='benthic_attribute_names', type=<ColInfoTypeEnum.default: 1>, column_label='benthic_attribute_names', column_align='right', column_width=None), ColInfo(var='benthic_attribute_ids', type=<ColInfoTypeEnum.default: 1>, column_label='benthic_attribute_ids', column_align='right', column_width=None), ColInfo(var='growth_forms', type=<ColInfoTypeEnum.default: 1>, column_label='growth_forms', column_align='right', column_width=None), ColInfo(var='regions', type=<ColInfoTypeEnum.default: 1>, column_label='regions', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x14555f710>, _spanners=Spanners([]), _heading=Heading(title='MERMAID corpus snapshot', subtitle='s3://coral-reef-training/mermaid/mermaid_confirmed_annotations.parquet', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x145612060>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x145612090>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1456120c0>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2

In [7]:
display(GT(schema_df).tab_header(title="Parquet schema"))

GT(_tbl_data=                    column                dtype  non_null  nunique
0                       id               string    415825   415825
1                 image_id               string    415825    16633
2                 point_id               string    415825   415825
3                      row                int64    415825     2570
4                      col                int64    415825     2744
5     benthic_attribute_id               string    415825      251
6   benthic_attribute_name               string    415825      251
7           growth_form_id               string    415825       12
8         growth_form_name               string     41832       11
9               updated_on  timestamp('UTC', 6)    415825   415825
10               region_id               string    415825        3
11             region_name               string    415825        3, _body=<great_tables._gt_data.Body object at 0x107e88d10>, _boxhead=Boxhead([ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None), ColInfo(var='dtype', type=<ColInfoTypeEnum.default: 1>, column_label='dtype', column_align='left', column_width=None), ColInfo(var='non_null', type=<ColInfoTypeEnum.default: 1>, column_label='non_null', column_align='right', column_width=None), ColInfo(var='nunique', type=<ColInfoTypeEnum.default: 1>, column_label='nunique', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x107e88ce0>, _spanners=Spanners([]), _heading=Heading(title='Parquet schema', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1456128d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x145612840>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x145612a50>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_i

## Corpus statistics by group

In [8]:
def group_stats(table, group_col: str) -> pd.DataFrame:
    total = table.count().execute()
    total_images = table.image_id.nunique().execute()
    grouped = table.group_by(group_col)
    return (
        grouped.aggregate(
            annotations=ibis._.count(),
            images=ibis._.image_id.nunique(),
        )
        .mutate(
            annotation_fraction=ibis._.annotations / total,
            image_fraction=ibis._.images / total_images,
        )
        .order_by(ibis.desc("annotations"), group_col)
        .to_pandas()
    )


def show_group_section(table, group_col: str, title: str) -> pd.DataFrame:
    stats = group_stats(table, group_col)
    display(
        GT(stats.head(30))
        .tab_header(title=title, subtitle=f"{len(stats)} groups · top 30 shown")
        .fmt_percent(columns=["annotation_fraction", "image_fraction"], decimals=2)
        .fmt_number(columns=["annotations", "images"], decimals=0)
    )
    stats.head(30).hvplot.barh(
        x=group_col,
        y="annotations",
        height=max(320, 18 * min(len(stats), 30)),
        width=720,
        title=title,
        xlabel="annotations",
        ylabel="",
        invert=True,
    )
    return stats

### Class distribution

In [9]:
class_stats = show_group_section(ann, "benthic_attribute_name", "Class distribution")

GT(_tbl_data=      benthic_attribute_name  annotations  images  annotation_fraction  \
0                 Macroalgae        83107   11902             0.199861   
1                       Sand        47209    7558             0.113531   
2                     Rubble        45419    8816             0.109226   
3             Bare substrate        36172    5518             0.086989   
4                 Turf algae        35192    6236             0.084632   
5   Crustose coralline algae        30345    7434             0.072975   
6                    Porites        20691    4394             0.049759   
7                 Hard coral        18668    4915             0.044894   
8                 Soft coral        15324    3513             0.036852   
9                   Acropora        10788    1964             0.025944   
10                Dead coral         8659    1680             0.020824   
11                 Sargassum         7707    1129             0.018534   
12                      Tape         5574    2138             0.013405   
13                   Galaxea         3704     522             0.008908   
14             Cyanobacteria         2981    1259             0.007169   
15               Pocillopora         2923    1222             0.007029   
16     Dead coral with algae         2871     461             0.006904   
17             Porites lutea         2777     610             0.006678   
18                 Montipora         2700    1004             0.006493   
19                 Millepora         2404     899             0.005781   
20                    Sponge         1960     840             0.004714   
21                      Rock         1623     319             0.003903   
22                  Seagrass         1520     490             0.003655   
23                     Other         1258     475             0.003025   
24                     Xenia         1227     118             0.002951   
25            Porites lobata         1214     377             0.002919   
26                Echinopora         1160     311             0.002790   
27                   Isopora         1074     311             0.002583   
28                Goniastrea         1049     545             0.002523   
29                 Platygyra          936     367             0.002251   

    image_fraction  
0         0.715565  
1         0.454398  
2         0.530031  
3         0.331750  
4         0.374917  
5         0.446943  
6         0.264174  
7         0.295497  
8         0.211207  
9         0.118079  
10        0.101004  
11        0.067877  
12        0.128540  
13        0.031383  
14        0.075693  
15        0.073468  
16        0.027716  
17        0.036674  
18        0.060362  
19        0.054049  
20        0.050502  
21        0.019179  
22        0.029460  
23        0.028558  
24        0.007094  
25        0.022666  
26        0.018698  
27        0.018698  
28        0.032766  
29        0.022065  , _body=<great_tables._gt_data.Body object at 0x143ef5f40>, _boxhead=Boxhead([ColInfo(var='benthic_attribute_name', type=<ColInfoTypeEnum.default: 1>, column_label='benthic_attribute_name', column_align='left', column_width=None), ColInfo(var='annotations', type=<ColInfoTypeEnum.default: 1>, column_label='annotations', column_align='right', column_width=None), ColInfo(var='images', type=<ColInfoTypeEnum.default: 1>, column_label='images', column_align='right', column_width=None), ColInfo(var='annotation_fraction', type=<ColInfoTypeEnum.default: 1>, column_label='annotation_fraction', column_align='right', column_width=None), ColInfo(var='image_fraction', type=<ColInfoTypeEnum.default: 1>, column_label='image_fraction', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x107e88ce0>, _spanners=Spanners([]), _heading=Heading(title='Class distribution', subtitle='251 groups · top 30 shown', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1456618e0>

### Region coverage

In [10]:
region_stats = show_group_section(ann, "region_name", "Region coverage")

GT(_tbl_data=            region_name  annotations  images  annotation_fraction  \
0  Central Indo-Pacific       273500   10940             0.657729   
1  Western Indo-Pacific       138400    5536             0.332832   
2     Tropical Atlantic         3925     157             0.009439   

   image_fraction  
0        0.657729  
1        0.332832  
2        0.009439  , _body=<great_tables._gt_data.Body object at 0x145599580>, _boxhead=Boxhead([ColInfo(var='region_name', type=<ColInfoTypeEnum.default: 1>, column_label='region_name', column_align='left', column_width=None), ColInfo(var='annotations', type=<ColInfoTypeEnum.default: 1>, column_label='annotations', column_align='right', column_width=None), ColInfo(var='images', type=<ColInfoTypeEnum.default: 1>, column_label='images', column_align='right', column_width=None), ColInfo(var='annotation_fraction', type=<ColInfoTypeEnum.default: 1>, column_label='annotation_fraction', column_align='right', column_width=None), ColInfo(var='image_fraction', type=<ColInfoTypeEnum.default: 1>, column_label='image_fraction', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x107e88ce0>, _spanners=Spanners([]), _heading=Heading(title='Region coverage', subtitle='3 groups · top 30 shown', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x145612b10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x145613710>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x145611a90>, _formats=[<great_tables._gt_data.FormatInfo object at 0x145611490>, <great_tables._gt_data.FormatInfo object at 0x1456101d0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_wi

### Benthic attribute 

In [11]:
benthic_stats = show_group_section(ann, "benthic_attribute_name", "Benthic attribute name")

GT(_tbl_data=      benthic_attribute_name  annotations  images  annotation_fraction  \
0                 Macroalgae        83107   11902             0.199861   
1                       Sand        47209    7558             0.113531   
2                     Rubble        45419    8816             0.109226   
3             Bare substrate        36172    5518             0.086989   
4                 Turf algae        35192    6236             0.084632   
5   Crustose coralline algae        30345    7434             0.072975   
6                    Porites        20691    4394             0.049759   
7                 Hard coral        18668    4915             0.044894   
8                 Soft coral        15324    3513             0.036852   
9                   Acropora        10788    1964             0.025944   
10                Dead coral         8659    1680             0.020824   
11                 Sargassum         7707    1129             0.018534   
12                      Tape         5574    2138             0.013405   
13                   Galaxea         3704     522             0.008908   
14             Cyanobacteria         2981    1259             0.007169   
15               Pocillopora         2923    1222             0.007029   
16     Dead coral with algae         2871     461             0.006904   
17             Porites lutea         2777     610             0.006678   
18                 Montipora         2700    1004             0.006493   
19                 Millepora         2404     899             0.005781   
20                    Sponge         1960     840             0.004714   
21                      Rock         1623     319             0.003903   
22                  Seagrass         1520     490             0.003655   
23                     Other         1258     475             0.003025   
24                     Xenia         1227     118             0.002951   
25            Porites lobata         1214     377             0.002919   
26                Echinopora         1160     311             0.002790   
27                   Isopora         1074     311             0.002583   
28                Goniastrea         1049     545             0.002523   
29                 Platygyra          936     367             0.002251   

    image_fraction  
0         0.715565  
1         0.454398  
2         0.530031  
3         0.331750  
4         0.374917  
5         0.446943  
6         0.264174  
7         0.295497  
8         0.211207  
9         0.118079  
10        0.101004  
11        0.067877  
12        0.128540  
13        0.031383  
14        0.075693  
15        0.073468  
16        0.027716  
17        0.036674  
18        0.060362  
19        0.054049  
20        0.050502  
21        0.019179  
22        0.029460  
23        0.028558  
24        0.007094  
25        0.022666  
26        0.018698  
27        0.018698  
28        0.032766  
29        0.022065  , _body=<great_tables._gt_data.Body object at 0x143ef5f40>, _boxhead=Boxhead([ColInfo(var='benthic_attribute_name', type=<ColInfoTypeEnum.default: 1>, column_label='benthic_attribute_name', column_align='left', column_width=None), ColInfo(var='annotations', type=<ColInfoTypeEnum.default: 1>, column_label='annotations', column_align='right', column_width=None), ColInfo(var='images', type=<ColInfoTypeEnum.default: 1>, column_label='images', column_align='right', column_width=None), ColInfo(var='annotation_fraction', type=<ColInfoTypeEnum.default: 1>, column_label='annotation_fraction', column_align='right', column_width=None), ColInfo(var='image_fraction', type=<ColInfoTypeEnum.default: 1>, column_label='image_fraction', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x107e88ce0>, _spanners=Spanners([]), _heading=Heading(title='Benthic attribute name', subtitle='251 groups · top 30 shown', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x145660

### Growth forms

In [12]:
growth_form_stats = show_group_section(ann, "growth_form_name", "Growth forms")

GT(_tbl_data=    growth_form_name  annotations  images  annotation_fraction  image_fraction
0               None       373993   16579             0.899400        0.996753
1          Branching        13686    2329             0.032913        0.140023
2            Massive        12654    3726             0.030431        0.224013
3         Encrusting         4559    1662             0.010964        0.099922
4           Columnar         3227     345             0.007760        0.020742
5         Submassive         2706     744             0.006508        0.044730
6   Plates or tables         2170     417             0.005219        0.025071
7           Digitate          984     351             0.002366        0.021103
8            Foliose          829     277             0.001994        0.016654
9          Corymbose          710     305             0.001707        0.018337
10    Mushroom coral          200     149             0.000481        0.008958
11       Arborescent          107      30             0.000257        0.001804, _body=<great_tables._gt_data.Body object at 0x14555eba0>, _boxhead=Boxhead([ColInfo(var='growth_form_name', type=<ColInfoTypeEnum.default: 1>, column_label='growth_form_name', column_align='left', column_width=None), ColInfo(var='annotations', type=<ColInfoTypeEnum.default: 1>, column_label='annotations', column_align='right', column_width=None), ColInfo(var='images', type=<ColInfoTypeEnum.default: 1>, column_label='images', column_align='right', column_width=None), ColInfo(var='annotation_fraction', type=<ColInfoTypeEnum.default: 1>, column_label='annotation_fraction', column_align='right', column_width=None), ColInfo(var='image_fraction', type=<ColInfoTypeEnum.default: 1>, column_label='image_fraction', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x145613830>, _spanners=Spanners([]), _heading=Heading(title='Growth forms', subtitle='12 groups · top 30 shown', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1456608f0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x145660f80>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x145661dc0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x14559af00>, <great_tables._gt_data.FormatInfo object at 0x1456615b0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_bor

### benthic_attribute_name × growth_form_name overlap

In [13]:
label_growth_grouped = ann.group_by(["benthic_attribute_name", "growth_form_name"])
label_growth = (
    label_growth_grouped.aggregate(
        annotations=ibis._.count(),
        images=ibis._.image_id.nunique(),
    )
    .mutate(annotation_fraction=ibis._.annotations / total_annotations)
    .order_by(ibis.desc("annotations"))
    .to_pandas()
)

forms_per_class_grouped = ann.group_by("benthic_attribute_name")
forms_per_class = (
    forms_per_class_grouped.aggregate(
        growth_forms=ibis._.growth_form_name.nunique(),
        annotations=ibis._.count(),
        images=ibis._.image_id.nunique(),
    )
    .order_by(ibis.desc("growth_forms"), ibis.desc("annotations"))
    .to_pandas()
)

classes_per_form_grouped = ann.group_by("growth_form_name")
classes_per_form = (
    classes_per_form_grouped.aggregate(
        benthic_attribute_names=ibis._.benthic_attribute_name.nunique(),
        annotations=ibis._.count(),
        images=ibis._.image_id.nunique(),
    )
    .order_by(ibis.desc("benthic_attribute_names"), ibis.desc("annotations"))
    .to_pandas()
)

overlap_summary = pd.DataFrame(
    [
        {
            "pairs": len(label_growth),
            "classes_with_forms": int((forms_per_class["growth_forms"] > 1).sum()),
            "forms_with_classes": int((classes_per_form["benthic_attribute_names"] > 1).sum()),
        }
    ]
)

In [14]:
display(GT(overlap_summary).tab_header(title="Label × growth-form overlap"))

GT(_tbl_data=   pairs  classes_with_forms  forms_with_classes
0    500                  58                  12, _body=<great_tables._gt_data.Body object at 0x145663380>, _boxhead=Boxhead([ColInfo(var='pairs', type=<ColInfoTypeEnum.default: 1>, column_label='pairs', column_align='right', column_width=None), ColInfo(var='classes_with_forms', type=<ColInfoTypeEnum.default: 1>, column_label='classes_with_forms', column_align='right', column_width=None), ColInfo(var='forms_with_classes', type=<ColInfoTypeEnum.default: 1>, column_label='forms_with_classes', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x144e45730>, _spanners=Spanners([]), _heading=Heading(title='Label × growth-form overlap', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x143ef5f40>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x145788980>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1457882f0>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), heading_background_color=OptionsInfo(scss=True, category='heading', type='value', value=None), heading_align=OptionsInfo(scss=True, category='heading', type='value', value='center'), heading_

In [15]:
display(
    GT(label_growth)
    .tab_header(title="benthic_attribute_name × growth_form_name", subtitle="top 40 pairs")
    .fmt_percent(columns=["annotation_fraction"], decimals=2)
    .fmt_number(columns=["annotations", "images"], decimals=0)
)

GT(_tbl_data=    benthic_attribute_name growth_form_name  annotations  images  \
0               Macroalgae             None        83102   11902   
1                     Sand             None        47209    7558   
2                   Rubble             None        45413    8815   
3           Bare substrate             None        36172    5518   
4               Turf algae             None        35192    6236   
..                     ...              ...          ...     ...   
495    Platygyra lamellina             None            1       1   
496        Cliona delitrix             None            1       1   
497               Faviidae             None            1       1   
498              Millepora          Foliose            1       1   
499            Pocillopora         Digitate            1       1   

     annotation_fraction  
0               0.199848  
1               0.113531  
2               0.109212  
3               0.086989  
4               0.084632  
..                   ...  
495             0.000002  
496             0.000002  
497             0.000002  
498             0.000002  
499             0.000002  

[500 rows x 5 columns], _body=<great_tables._gt_data.Body object at 0x107e88d10>, _boxhead=Boxhead([ColInfo(var='benthic_attribute_name', type=<ColInfoTypeEnum.default: 1>, column_label='benthic_attribute_name', column_align='left', column_width=None), ColInfo(var='growth_form_name', type=<ColInfoTypeEnum.default: 1>, column_label='growth_form_name', column_align='left', column_width=None), ColInfo(var='annotations', type=<ColInfoTypeEnum.default: 1>, column_label='annotations', column_align='right', column_width=None), ColInfo(var='images', type=<ColInfoTypeEnum.default: 1>, column_label='images', column_align='right', column_width=None), ColInfo(var='annotation_fraction', type=<ColInfoTypeEnum.default: 1>, column_label='annotation_fraction', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x144e45730>, _spanners=Spanners([]), _heading=Heading(title='benthic_attribute_name × growth_form_name', subtitle='top 40 pairs', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1457ba540>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1457ba720>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1457ba8d0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x143ef5f40>, <great_tables._gt_data.FormatInfo object at 0x1457bacc0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=Opti

In [16]:
display(
    GT(forms_per_class)
    .tab_header(title="Growth forms per benthic_attribute_name")
    .fmt_number(columns=["growth_forms", "annotations", "images"], decimals=0)
)

GT(_tbl_data=      benthic_attribute_name  growth_forms  annotations  images
0                    Porites            11        20691    4394
1                 Hard coral            11        18668    4915
2                  Montipora             8         2700    1004
3                     Pavona             8          515     246
4                   Acropora             7        10788    1964
..                       ...           ...          ...     ...
246  Montipora australiensis             0            1       1
247          Cliona delitrix             0            1       1
248      Agaricia agaricites             0            1       1
249           Favites abdita             0            1       1
250                     Clam             0            1       1

[251 rows x 4 columns], _body=<great_tables._gt_data.Body object at 0x145788b30>, _boxhead=Boxhead([ColInfo(var='benthic_attribute_name', type=<ColInfoTypeEnum.default: 1>, column_label='benthic_attribute_name', column_align='left', column_width=None), ColInfo(var='growth_forms', type=<ColInfoTypeEnum.default: 1>, column_label='growth_forms', column_align='right', column_width=None), ColInfo(var='annotations', type=<ColInfoTypeEnum.default: 1>, column_label='annotations', column_align='right', column_width=None), ColInfo(var='images', type=<ColInfoTypeEnum.default: 1>, column_label='images', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1457d6f30>, _spanners=Spanners([]), _heading=Heading(title='Growth forms per benthic_attribute_name', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x145789070>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x145788d70>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1457892e0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1457893a0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', 

In [17]:
display(
    GT(classes_per_form)
    .tab_header(title="benthic_attribute_name per growth_form_name")
    .fmt_number(columns=["benthic_attribute_names", "annotations", "images"], decimals=0)
)

GT(_tbl_data=    growth_form_name  benthic_attribute_names  annotations  images
0               None                      231       373993   16579
1         Encrusting                       59         4559    1662
2            Massive                       46        12654    3726
3         Submassive                       43         2706     744
4          Branching                       31        13686    2329
5            Foliose                       28          829     277
6   Plates or tables                       17         2170     417
7     Mushroom coral                       12          200     149
8           Columnar                       11         3227     345
9           Digitate                        8          984     351
10         Corymbose                        8          710     305
11       Arborescent                        6          107      30, _body=<great_tables._gt_data.Body object at 0x1456114c0>, _boxhead=Boxhead([ColInfo(var='growth_form_name', type=<ColInfoTypeEnum.default: 1>, column_label='growth_form_name', column_align='left', column_width=None), ColInfo(var='benthic_attribute_names', type=<ColInfoTypeEnum.default: 1>, column_label='benthic_attribute_names', column_align='right', column_width=None), ColInfo(var='annotations', type=<ColInfoTypeEnum.default: 1>, column_label='annotations', column_align='right', column_width=None), ColInfo(var='images', type=<ColInfoTypeEnum.default: 1>, column_label='images', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x144e478f0>, _spanners=Spanners([]), _heading=Heading(title='benthic_attribute_name per growth_form_name', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1457d6a50>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1457d6960>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1457d6930>, _formats=[<great_tables._gt_data.FormatInfo object at 0x14555fcb0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table'

In [18]:
top_pairs = label_growth.head(50).copy()
top_pairs["pair"] = top_pairs["benthic_attribute_name"] + " · " + top_pairs["growth_form_name"]

top_pairs.hvplot.barh(
    x="pair",
    y="annotations",
    height=520,
    width=900,
    title="Top 25 benthic_attribute_name × growth_form_name pairs",
    xlabel="annotations",
    ylabel="",
    invert=True,
)

:Bars   [pair]   (annotations)

## Train / val holdout

Rare-aware region-blocked holdout (`select_stratified_holdout_image_ids`): Pass 1 guarantees val coverage for rare `(region_name, benthic_attribute_name)` and `(region_name, growth_form_name)` pairs (2–9 images); Pass 2 fills the regional ~10% quota. Same `holdout_fraction=0.1`, `holdout_seed=42` as `MermaidDataset`.

Restart the kernel after editing `mermaid_dataset.py` so imports pick up changes (no `importlib.reload` needed).


In [19]:
from mermaidseg.datasets.mermaid.mermaid_dataset import (
    compute_split_quality,
    select_composite_holdout_image_ids,
    select_stratified_holdout_image_ids,
)

ann_pd = ann.to_pandas()
val_ids = select_stratified_holdout_image_ids(
    ann_pd,
    holdout_fraction=HOLDOUT_FRACTION,
    holdout_seed=HOLDOUT_SEED,
)
splits = pd.DataFrame({"image_id": ann_pd["image_id"].unique()})
splits["split"] = splits["image_id"].map(lambda i: "val" if i in val_ids else "train")
ann_split = ann.inner_join(ibis.memtable(splits), "image_id")

### Split quality diagnostics

Compare legacy composite-stratum holdout vs rare-aware holdout on the same annotation frame.


In [20]:
composite_val_ids = select_composite_holdout_image_ids(
    ann_pd,
    holdout_fraction=HOLDOUT_FRACTION,
    holdout_seed=HOLDOUT_SEED,
)
composite_quality = compute_split_quality(
    ann_pd,
    composite_val_ids,
    holdout_fraction=HOLDOUT_FRACTION,
)
rare_aware_quality = compute_split_quality(
    ann_pd,
    val_ids,
    holdout_fraction=HOLDOUT_FRACTION,
)


def _quality_row(name: str, val_set: set[str], quality: dict) -> dict:
    return {
        "strategy": name,
        "val_images": len(val_set),
        "rare_class_val_coverage": quality["rare_class_val_coverage"],
        "rare_growth_form_val_coverage": quality["rare_growth_form_val_coverage"],
        "classes_val_only": len(quality["classes_val_only"]),
        "classes_train_only": len(quality["classes_train_only"]),
    }


comparison = pd.DataFrame(
    [
        _quality_row("composite (legacy)", composite_val_ids, composite_quality),
        _quality_row("rare-aware", val_ids, rare_aware_quality),
    ]
)

display(
    GT(comparison)
    .tab_header(title="Holdout strategy comparison")
    .fmt_percent(
        columns=["rare_class_val_coverage", "rare_growth_form_val_coverage"],
        decimals=1,
    )
    .fmt_number(columns=["val_images", "classes_val_only", "classes_train_only"], decimals=0)
)

val_share = rare_aware_quality["val_share_by_region"]
display(
    GT(val_share)
    .tab_header(title="Val share by region (rare-aware)")
    .fmt_percent(columns=["val_fraction", "target_fraction"], decimals=1)
    .fmt_number(columns=["images", "val_images"], decimals=0)
)

missing = rare_aware_quality["missing_rare_class_pairs"]
if missing.empty:
    display(GT(pd.DataFrame([{"status": "All rare class pairs covered in val"}])))
else:
    display(
        GT(missing.head(30))
        .tab_header(title="Rare class pairs still missing val coverage")
        .fmt_number(columns=["images"], decimals=0)
    )

val_share.hvplot.bar(
    x="region_name",
    y="val_fraction",
    height=360,
    width=900,
    title="Val fraction by region (rare-aware)",
    rot=45,
)

GT(_tbl_data=             strategy  val_images  rare_class_val_coverage  \
0  composite (legacy)        1722                 0.372727   
1          rare-aware        1664                 1.000000   

   rare_growth_form_val_coverage  classes_val_only  classes_train_only  
0                            0.0                 2                  90  
1                            1.0                15                  44  , _body=<great_tables._gt_data.Body object at 0x15bc5bdd0>, _boxhead=Boxhead([ColInfo(var='strategy', type=<ColInfoTypeEnum.default: 1>, column_label='strategy', column_align='left', column_width=None), ColInfo(var='val_images', type=<ColInfoTypeEnum.default: 1>, column_label='val_images', column_align='right', column_width=None), ColInfo(var='rare_class_val_coverage', type=<ColInfoTypeEnum.default: 1>, column_label='rare_class_val_coverage', column_align='right', column_width=None), ColInfo(var='rare_growth_form_val_coverage', type=<ColInfoTypeEnum.default: 1>, column_label='rare_growth_form_val_coverage', column_align='right', column_width=None), ColInfo(var='classes_val_only', type=<ColInfoTypeEnum.default: 1>, column_label='classes_val_only', column_align='right', column_width=None), ColInfo(var='classes_train_only', type=<ColInfoTypeEnum.default: 1>, column_label='classes_train_only', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x15bc5bf80>, _spanners=Spanners([]), _heading=Heading(title='Holdout strategy comparison', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x15bc5b590>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x15bc5b560>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x15bc5b530>, _formats=[<great_tables._gt_data.FormatInfo object at 0x15bc5b410>, <great_tables._gt_data.FormatInfo object at 0x15bc5b4a0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', 

GT(_tbl_data=            region_name  images  val_images  val_fraction  target_fraction
0  Western Indo-Pacific    5536         554      0.100072              0.1
1  Central Indo-Pacific   10940        1094      0.100000              0.1
2     Tropical Atlantic     157          16      0.101911              0.1, _body=<great_tables._gt_data.Body object at 0x15bc5b6e0>, _boxhead=Boxhead([ColInfo(var='region_name', type=<ColInfoTypeEnum.default: 1>, column_label='region_name', column_align='left', column_width=None), ColInfo(var='images', type=<ColInfoTypeEnum.default: 1>, column_label='images', column_align='right', column_width=None), ColInfo(var='val_images', type=<ColInfoTypeEnum.default: 1>, column_label='val_images', column_align='right', column_width=None), ColInfo(var='val_fraction', type=<ColInfoTypeEnum.default: 1>, column_label='val_fraction', column_align='right', column_width=None), ColInfo(var='target_fraction', type=<ColInfoTypeEnum.default: 1>, column_label='target_fraction', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1560bcb60>, _spanners=Spanners([]), _heading=Heading(title='Val share by region (rare-aware)', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x15bc5b080>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x15bc5ad80>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x15bc5ae40>, _formats=[<great_tables._gt_data.FormatInfo object at 0x15bc5bf80>, <great_tables._gt_data.FormatInfo object at 0x15bc5b110>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table

status
All rare class pairs covered in val


:Bars   [region_name]   (val_fraction)

In [21]:
split_grouped = ann_split.group_by("split")
split_summary = (
    split_grouped.aggregate(
        annotations=ibis._.count(),
        images=ibis._.image_id.nunique(),
        benthic_attribute_names=ibis._.benthic_attribute_name.nunique(),
        regions=ibis._.region_name.nunique(),
    )
    .order_by("split")
    .to_pandas()
)

display(
    GT(split_summary)
    .tab_header(
        title="Image holdout split",
        subtitle=(
            f"fraction={HOLDOUT_FRACTION}, seed={HOLDOUT_SEED} · "
            f"train={(splits['split'] == 'train').sum()} images · "
            f"val={(splits['split'] == 'val').sum()} images"
        ),
    )
    .fmt_number(columns=["annotations", "images", "benthic_attribute_names", "regions"], decimals=0)
)

GT(_tbl_data=   split  annotations  images  benthic_attribute_names  regions
0  train       374225   14969                      236        3
1    val        41600    1664                      207        3, _body=<great_tables._gt_data.Body object at 0x156685370>, _boxhead=Boxhead([ColInfo(var='split', type=<ColInfoTypeEnum.default: 1>, column_label='split', column_align='left', column_width=None), ColInfo(var='annotations', type=<ColInfoTypeEnum.default: 1>, column_label='annotations', column_align='right', column_width=None), ColInfo(var='images', type=<ColInfoTypeEnum.default: 1>, column_label='images', column_align='right', column_width=None), ColInfo(var='benthic_attribute_names', type=<ColInfoTypeEnum.default: 1>, column_label='benthic_attribute_names', column_align='right', column_width=None), ColInfo(var='regions', type=<ColInfoTypeEnum.default: 1>, column_label='regions', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x15bc589b0>, _spanners=Spanners([]), _heading=Heading(title='Image holdout split', subtitle='fraction=0.1, seed=42 · train=14969 images · val=1664 images', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x15a3690a0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x15a369040>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x15a368ef0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x15a368e90>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=Options

In [22]:
def split_group_stats(group_col: str) -> pd.DataFrame:
    grouped = ann_split.group_by(["split", group_col])
    return (
        grouped.aggregate(
            annotations=ibis._.count(),
            images=ibis._.image_id.nunique(),
        )
        .order_by("split", ibis.desc("annotations"), group_col)
        .to_pandas()
    )


GROUP_COLS = [
    "region_name",
    "benthic_attribute_name",
    "benthic_attribute_id",
    "growth_form_name",
]
split_tables = {col: split_group_stats(col) for col in GROUP_COLS}

In [23]:
top_classes = ann_pd["benthic_attribute_name"].value_counts().head(15).index.tolist()
filtered = ann_split.filter(ann_split.benthic_attribute_name.isin(top_classes))
region_class_grouped = filtered.group_by(["region_name", "benthic_attribute_name", "split"])
region_class = (
    region_class_grouped.aggregate(annotations=ibis._.count())
    .order_by("region_name", "benthic_attribute_name", "split")
    .to_pandas()
)

GT(region_class.head(40)).tab_header(
    title="Top classes by region and split",
    subtitle="15 most frequent benthic_attribute_name values · top 40 rows",
).fmt_number(columns=["annotations"], decimals=0)

GT(_tbl_data=             region_name    benthic_attribute_name  split  annotations
0   Central Indo-Pacific                  Acropora  train         1889
1   Central Indo-Pacific                  Acropora    val          909
2   Central Indo-Pacific            Bare substrate  train        25751
3   Central Indo-Pacific            Bare substrate    val         1190
4   Central Indo-Pacific  Crustose coralline algae  train        13175
5   Central Indo-Pacific  Crustose coralline algae    val         1601
6   Central Indo-Pacific             Cyanobacteria  train          598
7   Central Indo-Pacific             Cyanobacteria    val           93
8   Central Indo-Pacific                Dead coral  train         7169
9   Central Indo-Pacific                Dead coral    val          690
10  Central Indo-Pacific                   Galaxea  train          117
11  Central Indo-Pacific                   Galaxea    val           31
12  Central Indo-Pacific                Hard coral  train        15276
13  Central Indo-Pacific                Hard coral    val         1166
14  Central Indo-Pacific                Macroalgae  train        62840
15  Central Indo-Pacific                Macroalgae    val         3360
16  Central Indo-Pacific                   Porites  train        13352
17  Central Indo-Pacific                   Porites    val         1271
18  Central Indo-Pacific                    Rubble  train        32872
19  Central Indo-Pacific                    Rubble    val         2441
20  Central Indo-Pacific                      Sand  train        31113
21  Central Indo-Pacific                      Sand    val         2322
22  Central Indo-Pacific                 Sargassum  train          668
23  Central Indo-Pacific                 Sargassum    val          576
24  Central Indo-Pacific                Soft coral  train         5407
25  Central Indo-Pacific                Soft coral    val          978
26  Central Indo-Pacific                      Tape  train         4686
27  Central Indo-Pacific                      Tape    val          575
28  Central Indo-Pacific                Turf algae  train        13219
29  Central Indo-Pacific                Turf algae    val         1239
30     Tropical Atlantic                  Acropora  train           13
31     Tropical Atlantic                  Acropora    val           11
32     Tropical Atlantic            Bare substrate  train          344
33     Tropical Atlantic            Bare substrate    val           33
34     Tropical Atlantic  Crustose coralline algae  train          137
35     Tropical Atlantic  Crustose coralline algae    val           31
36     Tropical Atlantic             Cyanobacteria  train          107
37     Tropical Atlantic             Cyanobacteria    val           16
38     Tropical Atlantic                Dead coral  train            2
39     Tropical Atlantic                Hard coral  train           48, _body=<great_tables._gt_data.Body object at 0x15a78c050>, _boxhead=Boxhead([ColInfo(var='region_name', type=<ColInfoTypeEnum.default: 1>, column_label='region_name', column_align='left', column_width=None), ColInfo(var='benthic_attribute_name', type=<ColInfoTypeEnum.default: 1>, column_label='benthic_attribute_name', column_align='left', column_width=None), ColInfo(var='split', type=<ColInfoTypeEnum.default: 1>, column_label='split', column_align='left', column_width=None), ColInfo(var='annotations', type=<ColInfoTypeEnum.default: 1>, column_label='annotations', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x15ba784d0>, _spanners=Spanners([]), _heading=Heading(title='Top classes by region and split', subtitle='15 most frequent benthic_attribute_name values · top 40 rows', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x15a8009e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x15a800a10>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tabl

In [24]:
split_group_stats("benthic_attribute_name").hvplot.bar(
    x="benthic_attribute_name",
    y="annotations",
    by="split",
    stacked=False,
    rot=45,
    height=420,
    width=900,
    title="Class annotations by holdout split",
    xlabel="",
    ylabel="annotations",
)

:Bars   [benthic_attribute_name,split]   (annotations)

## Optional: write local snapshot tables

In [25]:
# out = Path("artifacts/mermaid_snapshot")
# out.mkdir(parents=True, exist_ok=True)
# summary_df.to_csv(out / "summary.csv", index=False)
# for name, stats in {
#     "class_stats": class_stats,
#     "region_stats": region_stats,
#     "benthic_id_stats": benthic_id_stats,
#     "growth_form_stats": growth_form_stats,
#     "label_growth": label_growth,
#     "forms_per_class": forms_per_class,
#     "classes_per_form": classes_per_form,
# }.items():
#     stats.to_csv(out / f"{name}.csv", index=False)
# split_summary.to_csv(out / "holdout_split_summary.csv", index=False)
# for col, table in split_tables.items():
#     table.to_csv(out / f"holdout_by_{col}.csv", index=False)
# region_class_pivot.to_csv(out / "region_class_val_share.csv", index=False)